In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
META = "data/cox2_th_final_missing_05_13_02/output/cox2_th-meta-clean.tsv"
#META = 'data/cox2_th_final_missing_05_13_02/primer-trim/metadata.tsv'
EXCLUDE_REFSEQ = False

In [3]:
df = pd.read_csv(META, sep="\t")

if EXCLUDE_REFSEQ:
    df = df[~df["accession"].str.contains("_", na=False)]

counts = df.groupby("tax_id").size().reset_index(name="n_sequences")

fig = px.histogram(
    counts,
    x="n_sequences",
    nbins=50,
    labels={"n_sequences": "Sequences per species", "count": "Number of species"},
    title=f"Distribution of sequences per species{'  (RefSeq excluded)' if EXCLUDE_REFSEQ else ''}",
)
fig.update_layout(yaxis_title="Number of species", bargap=0.05)
fig.show()

In [4]:
def report_redundancy(df: pd.DataFrame) -> None:
    n_sequences = len(df)
    n_species = df["tax_id"].nunique()
    counts = df.groupby("tax_id").size()
    log_counts = counts.apply(np.log)
    gmean = np.exp(log_counts.mean())
    gstd = np.exp(log_counts.std())
    p90 = counts.quantile(0.90)
    print(f"Species:    {n_species}")
    print(f"Sequences:  {n_sequences}")
    print(f"Redundancy: {gmean:.2f} (×÷ {gstd:.2f})  (geometric mean sequences per species)")
    print(f"90th pct:   {p90:.0f} sequences per species")

report_redundancy(df)

# ── Pipeline redundancy across stages ─────────────────────────────────────────

BASE = "data/cox2_th_final_missing_05_13_02"

STAGES = [
    ("Download",              f"{BASE}/seq-meta.csv",                  ","),
    ("in-silico PCR",                  f"{BASE}/primer-trim/metadata.tsv",      "\t"),
    ("Metadata Filter",              f"{BASE}/seq-meta-filtered.tsv",         "\t"),
    ("Dereplicate",           f"{BASE}/dereplicate/metadata.tsv",      "\t"),
    ("Quality Filter",        f"{BASE}/quality-filter/metadata.tsv",   "\t"),
    ("Trim Alignment",        f"{BASE}/alignment/metadata-trim.tsv",   "\t"),
    ("Dereplicate Alignment", f"{BASE}/alignment/metadata.tsv",        "\t"),
    ("Gappiness",             f"{BASE}/gappiness/metadata.tsv",        "\t"),
    ("Mislabel Detection",    f"{BASE}/output/cox2_th-meta-clean.tsv", "\t"),
]

QUANTILES = [0.75, 0.85, 0.90, 0.95]

def _stage_stats(path, sep):
    d = pd.read_csv(path, sep=sep)
    if EXCLUDE_REFSEQ:
        d = d[~d["accession"].str.contains("_", na=False)]
    counts = d.groupby("tax_id").size()
    return {f"p{int(q*100)}": counts.quantile(q) for q in QUANTILES}

rows = [{"stage": s, **_stage_stats(p, sep)} for s, p, sep in STAGES]
pipeline_df = pd.DataFrame(rows)

fig2 = go.Figure()
for q in QUANTILES:
    col = f"p{int(q*100)}"
    fig2.add_trace(go.Scatter(
        x=pipeline_df["stage"],
        y=pipeline_df[col],
        name=f"{int(q*100)}th percentile",
        mode="lines+markers",
    ))

axis_style = dict(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", tickcolor="black",
    tickfont=dict(color="black"),
    title_font=dict(color="black"),
)
fig2.update_layout(
    width=800, height=450,
    title=dict(
        text=f"Sequence redundancy across pipeline stages{'  (RefSeq excluded)' if EXCLUDE_REFSEQ else ''}",
        font=dict(color="black"),
    ),
    font=dict(color="black"),
    xaxis=dict(tickangle=-30, **axis_style),
    yaxis=dict(title="Sequences per species", showgrid=True, gridcolor="lightgrey", **axis_style),
    plot_bgcolor="white",
    legend=dict(font=dict(color="black")),
)
fig2.show()

Species:    749
Sequences:  1327
Redundancy: 1.37 (×÷ 1.76)  (geometric mean sequences per species)
90th pct:   3 sequences per species


In [6]:
fig2.write_image("seq-per-species.svg")
pipeline_df.to_csv("redundancy-by-stage.csv", index=False, float_format="%.1f")

/tmp/ipykernel_3299015/4136857428.py:1: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


